# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-03_slm_vs_llm.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker — see the [course README](https://github.com/justkrismanohar/fine-tuning-build/blob/main/README.md).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day1/T1-03_slm_vs_llm.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day1"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day1" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


Both replies are scripted. OLMo’s canned C2 line is a bit longer; Phi’s is shorter. That stands in for ‘two models, same prompt.’ Real latency/quality is the Docker track. Still fill the cost card: Colab this afternoon is free; a public API still leaves the building.


# T1-03 Measuring two small models on one task

**Time:** 12:45–13:45 &nbsp;|&nbsp; **Evidence:** scorecard (`logs/t103_*.json`) + cost card (`cards/cost_card.md`)

**Notebook learning outcomes**

- Score OLMo and Phi on the same frozen CivicDesk screening set. Only live replies count.
- Report acceptable answers, correct decisions, tokens, latency, and cost per acceptable answer, with every assumption written down.
- Treat this as **two small open models, measured on one task** — not a small-versus-frontier ranking.

The filename still says SLM vs LLM. That is a leftover. Both classroom models are small.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Ministry link

**CPO, TTDF, Finance, THA** asked for Government-controlled hosting. The cost card is how you argue for that without inventing a ministry budget.

**Labour / Health / Trade** will be asked “why not just ChatGPT?” — privacy and cost-per-acceptable-answer are the adult reply.

This lab is **N11** and **N7**. OLMo 2 1B Instruct and Phi-4 Mini are both small open-weight models. You measure them on one frozen screening task. A hosted API is optional, later, and only on this fiction. Keep the same `BACKEND` you set in T1-01 (re-run that cell if you opened a new kernel).

## Overnight readers

T1-01 and T1-02 already showed tokens, templates and temperature. Today you do not re-teach that. You collect evidence: a frozen answer key, the same prompt to both models, a scorecard, then cost with the rates written down.


In [ ]:
# [RUN] same C2 line as a quick first look. Print mode and usage under each reply.
import civicdesk_lab as lab
# Re-apply T1-01 backend if this is a new kernel:
lab.use_backend("demo")  # Colab track — scripted OLMo vs Phi wording

prompt = (
    "CivicDesk. Screen this fictional line against criterion C2 "
    "(recognised diploma in records management or equivalent): "
    "'two-year diploma in library studies, 2019'. "
    "Reply with Meets / Does not meet / Human review required and a quote. Do not hire."
)
messages = [
    {"role": "system", "content": "Classroom assistant. Quote or refuse. No hiring."},
    {"role": "user", "content": prompt},
]
cfg = lab.endpoint_config()
olmo = lab.chat(messages, model=cfg["model"], max_tokens=96)
phi = lab.chat(messages, model=cfg["comparison_model"], max_tokens=96)
print("OLMo mode=", olmo["mode"], "usage=", olmo.get("usage"), olmo["latency_ms"], "ms")
print(olmo["text"][:400], "\n")
print("Phi  mode=", phi["mode"], "usage=", phi.get("usage"), phi["latency_ms"], "ms")
print(phi["text"][:400])


### Teaching note

A fluent paragraph is not evidence. The rest of this notebook uses a frozen set where “right” is defined before either model runs.

Neither model is promised to pass. A 1B model can fail this task completely; the scorecard will show that in numbers instead of impressions.


## What counts as a measurement here

Only replies with `mode == "live"` are measured. A demo or fallback reply is labelled as one and is **never** scored for accuracy, tokens, latency or cost.

**Counting convention:** Input tokens are the exact token IDs the model received after its own chat template was applied, including the system text, role markers and the start-of-reply marker. Output tokens are the new token IDs it generated, up to and including the first end-of-reply token. Each model uses its own tokenizer, so the same text gives different counts.

If the room is on vLLM instead of the container’s hf server, record `engine` and do not mix engines inside one comparison: the two servers count by their own rules.

If the GPU cannot hold both models, run the whole OLMo pass, restart with `CIVICDESK_LOAD_MODELS=phi`, then Phi. Record that split in `run_info`.


In [ ]:
# [RUN] Record the setup. Only live replies are measured.
import json, urllib.request, hashlib
from datetime import datetime, timezone
from pathlib import Path
import civicdesk_eval as ev

cfg = lab.endpoint_config()

def health(base_url):
    try:
        with urllib.request.urlopen(base_url.rstrip("/") + "/health", timeout=5) as r:
            return json.loads(r.read().decode())
    except Exception as e:
        return {"ok": False, "error": str(e)}

run_info = {
    "olmo": {"model": cfg["model"], "health": health(cfg["olmo_url"])},
    "phi": {"model": cfg["comparison_model"], "health": health(cfg["phi_url"])},
    "backend": cfg.get("backend"),
    "engine": cfg.get("engine"),
    "mode": cfg.get("mode"),
    "max_tokens": 96,
    "decoding": "greedy (server ignores temperature unless it is > 0)",
    "system_prompt_sha256": hashlib.sha256(ev.SCREEN_SYSTEM.encode()).hexdigest()[:16],
    "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "passes": "both servers at once",  # [CHANGE] or "sequential: OLMo then Phi after restart"
}
print(json.dumps(run_info, indent=2)[:2000])
MEASURED = (
    cfg.get("mode") == "live"
    and bool(run_info["olmo"]["health"].get("ok"))
    and bool(run_info["phi"]["health"].get("ok"))
)
if not MEASURED:
    print("=" * 60)
    print("NOT MEASURED: demo or fallback")
    print("=" * 60)
    print("You can still load the frozen set and fill the cost card with 'not available'.")


## The task: may this fictional draft go out?

A CivicDesk draft reply must be **BLOCK**ed if any rule applies. Otherwise it may **GO**.

- **R1:** it contains a CivicDesk personal ID (`CDID-` plus six digits). Ticket numbers (`TKT-` plus five digits) are fine.
- **R2:** it cites a circular that is not approved. Approved: Circular 12 of 2025 and Circular 7 of 2026.
- **R3:** it says CivicDesk has reset or will reset a password, or has unlocked or will unlock an account. “A staff member will review your request” is fine.
- **R4:** it tells an applicant they are hired, selected, shortlisted or ranked. “Your application was received and a person will review it” is fine.

The model helped write the wording. The rules decided the right answers, and a person checked them. If we had asked a model to label the cases and then scored models against those labels, we would be grading the test with the test.


In [ ]:
# [RUN] The frozen course set. Labels come from the rules and a person checked them.
rec = ev.load_frozen(Path("data/t103_release_cases_v1.json"))
cases = rec["cases"]
print(rec["n_cases"], "cases,", rec["n_block"], "BLOCK |", rec["review"], "| sha", rec["sha256"][:12])
print("rules version", rec["rules_version"], "| label source:", rec["label_source"])
for c in cases[:3]:
    print(c["case_id"], c["label"], c["rules"])
    print("  ", c["draft_reply"])
N_CASES = len(cases)  # [CHANGE] set to 12 if time is short; uses the first 12 by case ID
cases_run = sorted(cases, key=lambda c: c["case_id"])[:N_CASES]
print("scoring", len(cases_run), "cases")


In [ ]:
# [OPTIONAL] build a practice set. Label every output "practice, not the course set".
plan = ev.build_case_plan(seed=99)
practice, dropped = ev.build_cases(plan)  # template filler unless you pass a live chat_fn
print("PRACTICE, not the course set:", len(practice), "kept,", len(dropped), "dropped")
for d in dropped[:5]:
    print(" dropped", d["case_id"], d.get("drop_reason"))
prac_path = Path("logs/t103_practice_unreviewed.json")
ev.freeze(practice, prac_path, generator={"seed": 99, "note": "practice, not the course set"}, reviewed_by="")
print("wrote", prac_path, "(unreviewed; load with allow_unreviewed=True)")


In [ ]:
# [RUN] one token check on a single case. Tokenizer only — no weights in this kernel.
case0 = cases_run[0]
msgs = ev.screen_messages(case0)
print("Counting convention:\n", ev.COUNTING_CONVENTION, "\n")

def template_len(messages, source):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(source, local_files_only=True, trust_remote_code=True)
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=True)
    if hasattr(ids, "tolist"):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]
    return len(ids)

olmo_dir = Path(lab.tokenizer_source())
print("naive word split of the user text:", len(case0["draft_reply"].split()), "(not the real count)")
if MEASURED:
    probe = lab.chat(msgs, model=cfg["model"], max_tokens=16)
    print("server prompt_tokens:", (probe.get("usage") or {}).get("prompt_tokens"), "← real count for this request")
    print("server completion_tokens:", (probe.get("usage") or {}).get("completion_tokens"), "← real output count")
    print("re-tokenized decoded reply length is usually one less (EOS is not in the text)")
    try:
        print("OLMo apply_chat_template length:", template_len(msgs, str(olmo_dir)), "← should match prompt_tokens")
    except Exception as e:
        print("tokenizer check skipped:", e)
else:
    print("NOT MEASURED: skip live token check")


In [ ]:
# [RUN] Same cases, same prompt, same settings. One request at a time. No third model.
def chat_fn(messages, model, max_tokens=96):
    return lab.chat(messages, model=model, max_tokens=max_tokens, timeout=120)

results = {}
if not MEASURED:
    print("NOT MEASURED: demo or fallback — not scoring")
    for key in ("olmo", "phi"):
        results[key] = {"rows": [], "summary": ev.aggregate([])}
else:
    for key, model in [("olmo", cfg["model"]), ("phi", cfg["comparison_model"])]:
        print("===", key, model)
        rows = ev.run_eval(cases_run, model, chat_fn, max_tokens=96, warmup=True)
        results[key] = {"rows": rows, "summary": ev.aggregate(rows)}
        for r in rows:
            tag = "warm" if not r["scored"] else r["case_id"]
            print(f"  {tag:8} mode={r['mode']} dec={r['decision']} acc={r['acceptable']} in={r['prompt_tokens']} out={r['completion_tokens']}")
        print(key, {k: results[key]["summary"][k] for k in ("rows_live", "acceptable", "decision_correct", "measured")})


In [ ]:
# [RUN] scorecard: both models, live rows only in the accuracy columns.
from datetime import datetime
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
run_info["n_cases_scored"] = len(cases_run)
run_info["frozen_sha256"] = rec["sha256"]
run_info["frozen_review"] = rec["review"]
run_info["rules_version"] = rec["rules_version"]
run_info["measured"] = bool(MEASURED)
ev.print_side_by_side(results["olmo"]["summary"], results["phi"]["summary"])
paths = []
for key in ("olmo", "phi"):
    csv_p, json_p = ev.write_scorecard(
        results[key]["rows"],
        results[key]["summary"],
        Path("logs") / f"t103_{key}_{ts}",
        run_info={**run_info, "model_key": key, "counting_convention": ev.COUNTING_CONVENTION},
    )
    paths.extend([csv_p, json_p])
    print("wrote", csv_p)
print("24 cases is a small sample; a difference of a few cases may be noise.")


## How to read the scorecard

24 cases is a small sample. The 95% Wilson interval is wide on purpose. A difference of a few cases may be noise. Latency is for **this** hardware and these settings. Nothing here ranks the models in general, and nothing here is a Ministry document.

A bare `GO` is a readable decision and is not an acceptable answer. Format, named rule and quote all have to hold.


## Cost: tokens are not prices

The local servers have no price per token. Tokens measure work and context used. They become money only through a rate somebody charges.

**Hosted API (illustrative)**

`api_cost = input_tokens × input_rate_per_million / 1,000,000 + output_tokens × output_rate_per_million / 1,000,000`

Use the provider’s own reported usage and its published rate on a stated date. Caching discounts, batch discounts, search grounding, taxes and minimum spends are excluded.

**Self-hosted OLMo and Phi** — two different numbers; show both.

- **Marginal active estimate:** hours = the sum of `generation_ms` for this batch. “What did these answers use while the GPU was busy.”
- **Billed allocation:** hours = the time the instance was allocated and billed, including start-up, loading, warm-up and idle. “What did we pay.”
- **Shared host:** split the billed cost with one stated rule (by active generation seconds is the default). Check that the shares add up to the bill. Never charge the full host to each model.

`cost_per_acceptable = total cost of every attempt in the batch / number of acceptable answers`

Every attempt counts: the warm-up, every scored case and any retry. Zero acceptable answers: print **undefined**, never 0, never a division error.

Staff time, monitoring, storage, support, security and retraining are out of scope for this cell.


In [ ]:
# [CHANGE] Fill only figures you have, with a source and date. Leave None if unknown.
currency = "USD"
api = {"input_rate_per_million": None, "output_rate_per_million": None, "source": "", "date": ""}
host = {"hourly_rate": None, "billed_hours": None, "source": "", "date": ""}

s_o, s_p = results["olmo"]["summary"], results["phi"]["summary"]
active = {"olmo": s_o["attempt_generation_s_total"] or 0, "phi": s_p["attempt_generation_s_total"] or 0}
for key, s in [("olmo", s_o), ("phi", s_p)]:
    print(f"--- {key}: {s['acceptable']} acceptable of {s['rows_live']} live")
    if s["attempts_without_usage"]:
        print("warning:", s["attempts_without_usage"], "live attempts had no token counts; token cost is incomplete")
    if None not in (api["input_rate_per_million"], api["output_rate_per_million"]):
        c = ev.api_cost(s["attempt_input_tokens_total"], s["attempt_output_tokens_total"], api["input_rate_per_million"], api["output_rate_per_million"])
        per = ev.cost_per_acceptable(c, s["acceptable"])
        print("illustrative API cost:", round(c, 6), currency, "| per acceptable:", per if per is not None else "undefined: no acceptable answers in this batch")
    else:
        print("illustrative API cost: not available (no rate entered)")
    if host["hourly_rate"] is not None:
        marginal = ev.compute_cost((active[key] or 0) / 3600, host["hourly_rate"])
        per_m = ev.cost_per_acceptable(marginal, s["acceptable"])
        print("marginal active compute:", round(marginal, 4), currency, "| per acceptable:", per_m if per_m is not None else "undefined")
        if host["billed_hours"] is not None and sum(active.values()) > 0:
            share = active[key] / sum(active.values())
            billed = ev.compute_cost(host["billed_hours"], host["hourly_rate"], share)
            per_b = ev.cost_per_acceptable(billed, s["acceptable"])
            print(f"billed share ({share:.0%} by active seconds):", round(billed, 4), currency, "| per acceptable:", per_b if per_b is not None else "undefined")
    else:
        print("self-hosted compute: not available (no hourly rate entered)")
print("Included: warm-up and every attempt. Excluded: caching, discounts, taxes, staff time, storage, support.")


### Made-up numbers to show the arithmetic

These are **not** market prices.

- A batch has 25 requests (1 warm-up + 24 cases), 7,200 input tokens and 900 output tokens in total, and 15 acceptable answers.
- Hosted API at made-up rates of US$0.40 per million input tokens and US$1.60 per million output tokens: 7,200 × 0.40 / 1,000,000 = US$0.00288, plus 900 × 1.60 / 1,000,000 = US$0.00144, so US$0.00432. Per acceptable answer: US$0.00432 / 15 = US$0.000288.
- Self-hosted at a made-up US$1.00 an hour, 120 seconds of active generation: 120 / 3,600 × 1.00 = US$0.0333 marginal, or about US$0.0022 per acceptable answer.
- The same host was allocated for 45 minutes (US$0.75 billed) and shared with Phi, which used 180 active seconds. OLMo’s share by active seconds is 120 / 300 = 40%, so US$0.30, or US$0.02 per acceptable answer. Phi’s share is US$0.45. The two add up to US$0.75.
- If the batch had 0 acceptable answers, cost per acceptable answer is undefined, not US$0.

The billed figure is about ten times the marginal one here because of idle and loading time. Utilisation drives self-hosted cost.


## Optional: the same frozen cases on a hosted API

**Free tiers can use what you send.** Google’s own terms for unpaid Gemini use say it may use your prompts and responses to improve its products and that human reviewers may read them. Other providers’ rules differ, and they change. Send only the fictional CivicDesk cases from this notebook. Never send Ministry documents, Cabinet notes, résumés, personal data or anything you would not post publicly.

The main path never needs a key. Skip this section with the room.


### How to get a key (checked 12 November 2026 — re-check before you rely on it)

Free access may not cover a whole class (shared limits, daily caps, account eligibility) and may stop.

**Google AI Studio (Gemini API).** Create a key at https://aistudio.google.com/api-keys. Set `GEMINI_API_KEY` or `GOOGLE_API_KEY` (`GOOGLE_API_KEY` wins if both are set). Trinidad and Tobago is listed on the available-regions page. Unpaid use may be read by human reviewers — fictional cases only. OpenAI-compatible URL: `https://generativelanguage.googleapis.com/v1beta/openai/`. Docs: https://ai.google.dev/gemini-api/docs/api-key

**GroqCloud.** Key: https://console.groq.com/keys. Set `GROQ_API_KEY`. Example open-weight model: `openai/gpt-oss-20b`. Free-plan limits apply to the organisation. Base URL: `https://api.groq.com/openai/v1`. If replies come back empty, raise `max_tokens` or set `include_reasoning: false` and record the setting.

**OpenRouter.** Key: https://openrouter.ai/keys. A `:free` model ID is free; the serving provider is chosen unless you pin it. Pin in the next cell. Free-variant limits: 20 requests a minute; 50 a day below US$10 of credits ever. Each provider has its own data policy.


In [ ]:
# [OPTIONAL] key for this session only. Never printed, never saved.
import civicdesk_remote as remote
# [CHANGE] "groq" or "google" or "openrouter"
PROVIDER = "groq"
print(remote.set_key(PROVIDER))


In [ ]:
# [OPTIONAL] [CHANGE] one model ID per provider, written out. No router aliases.
REMOTE_MODELS = {
    "google": "gemini-3.5-flash-lite",
    "groq": "openai/gpt-oss-20b",
    "openrouter": "google/gemma-4-26b-a4b-it:free",
}
EXTRA = {}
if PROVIDER == "openrouter":
    EXTRA = {"provider": {"order": ["google-ai-studio"], "allow_fallbacks": False}}
if PROVIDER == "groq":
    EXTRA = {"include_reasoning": False}
print("provider", PROVIDER, "model", REMOTE_MODELS[PROVIDER], "extra", EXTRA)


In [ ]:
# [OPTIONAL] same frozen cases, sequential, paused for free-tier limits.
if not cases_run:
    print("load the course set first")
else:
    chat_remote = remote.make_chat_fn(PROVIDER, REMOTE_MODELS[PROVIDER], extra_body=EXTRA or None, pause_s=3)
    remote_rows = ev.run_eval(cases_run, REMOTE_MODELS[PROVIDER], chat_remote, max_tokens=96, warmup=True)
    remote_summary = ev.aggregate(remote_rows)
    print(remote_summary)
    print("usage_source sample:", remote_rows[0]["usage_source"] if remote_rows else None)
    print("Provider token counts are not comparable number-for-number with the local ones.")
    ev.write_scorecard(remote_rows, remote_summary, Path("logs") / f"t103_{PROVIDER}_{ts}", run_info={"provider": PROVIDER, "optional": True})


In [ ]:
# [OPTIONAL] two lines that are never added together. Re-check rates on the day.
print("Observed charge on the free tier: US$0 (for this run, on this account, on this date).")
paid_in = None   # [CHANGE] published input USD / million tokens, or None
paid_out = None  # [CHANGE] published output USD / million tokens, or None
if "remote_summary" not in globals():
    print("Paid-equivalent at the published rate on [date]: not available (hosted eval not run)")
elif remote_summary.get("attempt_input_tokens_total") and None not in (paid_in, paid_out):
    paid = ev.api_cost(remote_summary["attempt_input_tokens_total"], remote_summary["attempt_output_tokens_total"], paid_in, paid_out)
    print(f"Paid-equivalent at the published rate on [date]: {paid} USD")
else:
    print("Paid-equivalent at the published rate on [date]: not available")


A hosted model scoring higher on 24 fictional cases says nothing about Ministry documents, and a free tier being US$0 today is not a budget. Hosted models are not assumed to be better. The scorecard decides, case by case, and only for this task.


In [ ]:
# [OPTIONAL] drop keys from this process before you download the notebook.
remote.forget_keys()
print("keys forgotten for this process")


In [ ]:
# [CHANGE] write-up. Numbers come from the scorecard. Fill the two human fields.
s_o, s_p = results["olmo"]["summary"], results["phi"]["summary"]
writeup = {
    "olmo_acceptable_of_live": f"{s_o.get('acceptable')} of {s_o.get('rows_live')}",
    "phi_acceptable_of_live": f"{s_p.get('acceptable')} of {s_p.get('rows_live')}",
    "olmo_decisions_correct": s_o.get("decision_correct"),
    "phi_decisions_correct": s_p.get("decision_correct"),
    "measured": bool(s_o.get("measured") and s_p.get("measured")),
    "who_would_pay": "laptop / Ministry VM / MEA Azure / hosted API / unsure",
    "data_allowed_to_leave_our_control": "yes / no / only if fictional / unsure",
}
print(writeup)
lab.save_log("T1-03_scorecard", {
    "run_info": run_info,
    "olmo": results["olmo"]["summary"],
    "phi": results["phi"]["summary"],
    "writeup": writeup,
})


## Cost card (also on paper)

Open `cards/cost_card.md`. Copy the live figures from the scorecard. Leave a field blank when you have no rate. Burst vs standing inference is a Day 3 mark; today the billed-versus-marginal gap is the answer to “if we left the GPU on overnight, who notices?”

## Write-up

1. Using **your** scorecard, which model would you show a domain verifier for this screening task, and why? If `measured` is false, say what you cannot conclude.
2. Name one reason a closed API is cheaper *this afternoon* and one reason it is expensive for a core process.
3. Would this data be allowed to leave our control under our rules?
